# Forensic classification challenge

Use this notebook to compare a small set of classifiers and predict the most likely class for mystery samples.

To swap in a new challenge, upload replacement CSV files in the JupyterLite file browser. If you keep the same filenames and columns, no notebook edits are required.


In [ ]:
import piplite
await piplite.install(["python-dateutil", "pandas", "scikit-learn"])


In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


In [ ]:
LABEL_COLUMN = "class_label"
ID_COLUMN = "sample_id"
TRAINING_DATA_CANDIDATES = ["data/training_samples.csv", "../data/training_samples.csv"]
MYSTERY_DATA_CANDIDATES = ["data/mystery_samples.csv", "../data/mystery_samples.csv"]

def resolve_existing_path(candidates):
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path
    raise FileNotFoundError(f"None of the candidate paths exist: {candidates}")

TRAINING_DATA_PATH = resolve_existing_path(TRAINING_DATA_CANDIDATES)
MYSTERY_DATA_PATH = resolve_existing_path(MYSTERY_DATA_CANDIDATES)
TRAINING_DATA_PATH, MYSTERY_DATA_PATH


In [ ]:
training_df = pd.read_csv(TRAINING_DATA_PATH)
mystery_df = pd.read_csv(MYSTERY_DATA_PATH)
feature_columns = [column for column in training_df.columns if column not in {ID_COLUMN, LABEL_COLUMN}]

assert feature_columns, "No feature columns found in training data"
assert feature_columns == [column for column in mystery_df.columns if column != ID_COLUMN], "Training and mystery feature columns must match"

training_df.head()


In [ ]:
MODEL_CANDIDATES = {
    "logistic_regression": LogisticRegression(max_iter=2_000),
    "random_forest": RandomForestClassifier(n_estimators=250, random_state=42),
    "knn": KNeighborsClassifier(n_neighbors=5),
    "svc_rbf": SVC(kernel="rbf", probability=True),
}

def make_pipeline(model):
    return Pipeline([
        ("preprocess", ColumnTransformer([
            ("numeric", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), feature_columns),
        ])),
        ("model", model),
    ])

X_train = training_df[feature_columns]
y_train = training_df[LABEL_COLUMN]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []
for model_name, model in MODEL_CANDIDATES.items():
    scores = cross_validate(
        make_pipeline(model),
        X_train,
        y_train,
        cv=cv,
        scoring={"accuracy": "accuracy", "f1_macro": "f1_macro"},
        n_jobs=1,
    )
    results.append({
        "model": model_name,
        "mean_accuracy": scores["test_accuracy"].mean(),
        "mean_f1_macro": scores["test_f1_macro"].mean(),
    })

results_df = pd.DataFrame(results).sort_values(["mean_f1_macro", "mean_accuracy"], ascending=False).reset_index(drop=True)
results_df


In [ ]:
best_model_name = results_df.loc[0, "model"]
best_pipeline = make_pipeline(MODEL_CANDIDATES[best_model_name])
best_pipeline.fit(X_train, y_train)

in_sample_predictions = best_pipeline.predict(X_train)
print(classification_report(y_train, in_sample_predictions))
best_model_name


In [ ]:
mystery_predictions = mystery_df[[ID_COLUMN]].copy()
mystery_predictions["predicted_class"] = best_pipeline.predict(mystery_df[feature_columns])
mystery_predictions


## Next step

Use the predicted class labels as the treasure-hunt clue, or export `mystery_predictions` to CSV for the next stage.
